# BirdCLEF+ 2026 — EDA

## 目的
1. **クラス不均衡の把握** → 損失関数・サンプリング戦略を決定
2. **音声ファイルの特性把握** → ウィンドウ戦略・パディング量を決定
3. **Secondary labels の活用可能性確認** → soft label の重みを決定
4. **音声の視覚的確認** → スペクトログラムパラメータを決定

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
from IPython.display import Audio, display
from collections import Counter

warnings.filterwarnings('ignore')
COMP = '/kaggle/input/birdclef-2026'
SR   = 32000
print('データパス:', COMP)
print('ディレクトリ一覧:', os.listdir(COMP))

## 1. メタデータの基本確認

In [ ]:
df = pd.read_csv(f'{COMP}/train_metadata.csv')
print(f'行数: {len(df):,}  |  列数: {len(df.columns)}')
print('\nカラム一覧:')
print(df.dtypes)
df.head(5)

In [ ]:
# 欠損値の確認
miss = df.isnull().sum()
miss = miss[miss > 0]
print('欠損値のある列:')
print(miss if len(miss) else '  なし')

## 2. クラス分布 — 最重要

In [ ]:
counts = df['primary_label'].value_counts()
n_cls  = len(counts)

print(f'総クラス数: {n_cls}')
print(f'合計サンプル数: {len(df):,}')
print(f'\n--- サンプル数の統計 ---')
print(f'最小: {counts.min()}')
print(f'中央値: {counts.median():.0f}')
print(f'平均: {counts.mean():.1f}')
print(f'最大: {counts.max()}')
print(f'\n5サンプル以下のクラス数: {(counts <= 5).sum()}')
print(f'20サンプル以下のクラス数: {(counts <= 20).sum()}')
print(f'\n最少サンプル上位10種:')
print(counts.tail(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# ヒストグラム
axes[0].hist(counts.values, bins=50, edgecolor='white', color='steelblue')
axes[0].set_xlabel('クラス別サンプル数')
axes[0].set_ylabel('クラス数')
axes[0].set_title('クラス別サンプル数の分布')

# 上位30クラス
counts.head(30).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('上位30クラス (サンプル数多)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)

# 下位30クラス
counts.tail(30).plot(kind='bar', ax=axes[2], color='tomato')
axes[2].set_title('下位30クラス (サンプル数少) ← 要注意')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=90, labelsize=7)

plt.tight_layout()
plt.show()

# 不均衡率
imbalance_ratio = counts.max() / counts.min()
print(f'\n不均衡率 (max/min): {imbalance_ratio:.1f}x')
print('→ Focal Loss が有効な理由がわかる')

## 3. Secondary Labels の確認

In [ ]:
import ast

def parse_secondary(raw):
    if pd.isna(raw) or str(raw).strip() in ('[]', ''):
        return []
    try:
        return ast.literal_eval(str(raw))
    except:
        return []

df['sec_parsed'] = df['secondary_labels'].apply(parse_secondary)
df['n_secondary'] = df['sec_parsed'].apply(len)

has_sec = (df['n_secondary'] > 0).sum()
print(f'secondary labels を持つ行数: {has_sec:,} / {len(df):,} ({has_sec/len(df)*100:.1f}%)')
print(f'\nsecondary label 数の分布:')
print(df['n_secondary'].value_counts().sort_index().head(10).to_string())

# secondary として出現頻度が高い種
all_sec = [s for lst in df['sec_parsed'] for s in lst]
print(f'\n合計 secondary label 数: {len(all_sec):,}')
sec_counts = pd.Series(Counter(all_sec)).sort_values(ascending=False)
print('\nSecondary として頻出の上位20種:')
print(sec_counts.head(20).to_string())

## 4. 音声ファイルの特性

In [ ]:
# ランダムサンプリングして音声ファイルの長さを計測
# (全ファイルは時間がかかるので100件)
sample_df = df.sample(min(100, len(df)), random_state=42)
durations = []
errors    = []

for _, row in sample_df.iterrows():
    path = os.path.join(COMP, 'train_audio', row['filename'])
    if not os.path.exists(path):
        errors.append(path)
        continue
    try:
        dur = librosa.get_duration(path=path)
        durations.append(dur)
    except:
        errors.append(path)

dur_arr = np.array(durations)
print(f'サンプリング数: {len(durations)} 件 (エラー: {len(errors)} 件)')
print(f'\n音声長の統計 (秒):')
print(f'  最小: {dur_arr.min():.1f}s')
print(f'  中央値: {np.median(dur_arr):.1f}s')
print(f'  平均: {dur_arr.mean():.1f}s')
print(f'  最大: {dur_arr.max():.1f}s')
print(f'\n  5s 以下: {(dur_arr < 5).sum()} 件 → zero-padding が必要')
print(f'  5s ちょうど: {((dur_arr >= 4.9) & (dur_arr <= 5.1)).sum()} 件')
print(f'  5s 超: {(dur_arr > 5).sum()} 件')

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(dur_arr, bins=30, edgecolor='white', color='steelblue')
ax.axvline(5, color='red', linestyle='--', label='5秒 (1ウィンドウ)')
ax.set_xlabel('音声長 (秒)')
ax.set_ylabel('件数')
ax.set_title('学習音声ファイルの長さ分布')
ax.legend()
plt.tight_layout()
plt.show()

## 5. メルスペクトログラムの視覚確認

In [ ]:
# サンプル音声を複数種で可視化
# (よく出現する種 + 少ないサンプルの種 を比較)
top_species    = df['primary_label'].value_counts().head(3).index.tolist()
bottom_species = df['primary_label'].value_counts().tail(3).index.tolist()
sample_species = top_species + bottom_species

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.3)

for i, sp in enumerate(sample_species):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    row = df[df['primary_label'] == sp].iloc[0]
    path = os.path.join(COMP, 'train_audio', row['filename'])
    
    try:
        y, _ = librosa.load(path, sr=SR, duration=5.0, mono=True)
        if len(y) < SR * 5:
            y = np.pad(y, (0, SR * 5 - len(y)))
        mel = librosa.feature.melspectrogram(
            y=y, sr=SR, n_mels=128, n_fft=2048, hop_length=512, fmin=50, fmax=16000)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        librosa.display.specshow(mel_db, sr=SR, hop_length=512, x_axis='time',
                                  y_axis='mel', fmin=50, fmax=16000, ax=ax)
        label_type = '(多)' if sp in top_species else '(少)'
        n_samp = df[df['primary_label'] == sp].shape[0]
        ax.set_title(f'{sp} {label_type}\nn={n_samp}', fontsize=9)
    except Exception as e:
        ax.set_title(f'{sp}\n(読み込み失敗)', fontsize=9)
        ax.text(0.5, 0.5, str(e), transform=ax.transAxes, ha='center')

plt.suptitle('メルスペクトログラムの例\n(上段: サンプル数多い種 / 下段: サンプル数少ない種)', y=1.02)
plt.show()

## 6. テストサウンドスケープの確認

In [ ]:
test_dir   = f'{COMP}/test_soundscapes'
test_files = sorted(glob.glob(f'{test_dir}/*.ogg'))
print(f'テストファイル数: {len(test_files)}')

if test_files:
    test_durs = []
    for fp in test_files[:20]:   # 最大20件で推定
        try:
            test_durs.append(librosa.get_duration(path=fp))
        except:
            pass
    test_durs = np.array(test_durs)
    print(f'\nテスト音声長 (最大20件サンプル):')
    print(f'  最小: {test_durs.min():.0f}s')
    print(f'  最大: {test_durs.max():.0f}s')
    print(f'  平均: {test_durs.mean():.0f}s')

    total_windows = sum(int(d / 5) for d in test_durs) * (len(test_files) // max(len(test_durs), 1))
    print(f'\n推定総ウィンドウ数 (overlap なし): ~{total_windows:,}')
    print(f'推定総ウィンドウ数 (2.5s overlap): ~{total_windows * 2:,}')

In [ ]:
# テストサウンドスケープのスペクトログラム (先頭1ファイル・30秒分)
if test_files:
    y_test, _ = librosa.load(test_files[0], sr=SR, duration=30.0, mono=True)
    mel = librosa.feature.melspectrogram(
        y=y_test, sr=SR, n_mels=128, n_fft=2048, hop_length=512, fmin=50, fmax=16000)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    fig, ax = plt.subplots(figsize=(16, 4))
    librosa.display.specshow(mel_db, sr=SR, hop_length=512,
                              x_axis='time', y_axis='mel', fmin=50, fmax=16000, ax=ax)
    plt.colorbar(ax.images[0], ax=ax, format='%+2.0f dB')
    # 5秒区切りの縦線
    for t in range(5, 31, 5):
        ax.axvline(t, color='white', alpha=0.4, linestyle='--')
    ax.set_title(f'テストサウンドスケープ: {os.path.basename(test_files[0])} (先頭30秒)\n白線 = 5秒ウィンドウ境界')
    plt.tight_layout()
    plt.show()

## 7. Sample Submission の確認

In [ ]:
sub = pd.read_csv(f'{COMP}/sample_submission.csv')
species_cols = [c for c in sub.columns if c != 'row_id']
print(f'提出行数: {len(sub):,}')
print(f'種列数 (= 分類クラス数): {len(species_cols)}')
print(f'\nrow_id の例:')
print(sub['row_id'].head(5).tolist())

# row_id のパターン確認
parts = sub['row_id'].str.rsplit('_', n=1, expand=True)
parts.columns = ['filename_stem', 'second']
seconds = parts['second'].astype(int)
print(f'\n秒数の種類: {sorted(seconds.unique())[:20]} ...')
print(f'→ {seconds.min()}s 〜 {seconds.max()}s まで、5s 刻みで評価される')

In [ ]:
# 学習データの種 vs 提出要求の種の対応確認
train_species = set(df['primary_label'].unique())
sub_species   = set(species_cols)

only_in_train = train_species - sub_species
only_in_sub   = sub_species   - train_species
in_both       = train_species & sub_species

print(f'学習データの種数: {len(train_species)}')
print(f'提出要求の種数:   {len(sub_species)}')
print(f'両方にある種数:   {len(in_both)}')
print(f'学習のみにある種: {len(only_in_train)} (提出不要)')
print(f'提出のみにある種: {len(only_in_sub)} ← 学習データがない!')

if only_in_sub:
    print(f'\n提出要求だが学習データなし (先頭10種):')
    print(sorted(only_in_sub)[:10])
    print('→ これらの種は予測スコアが常に 0.0 になる (Padded cmAP への影響を確認すること)')

## 8. EDA まとめ & 戦略への示唆

以下のセルを実行することでコンペ固有の数値が入ります。

In [ ]:
print('=' * 55)
print('EDA サマリー & configs への反映ポイント')
print('=' * 55)

print(f'\n[データ規模]')
print(f'  クラス数:     {n_cls}')
print(f'  総サンプル数: {len(df):,}')

print(f'\n[クラス不均衡]')
print(f'  最小サンプル: {counts.min()}')
print(f'  最大サンプル: {counts.max()}')
imb = counts.max() / counts.min()
print(f'  不均衡率:     {imb:.0f}x → Focal Loss 必須')
if (counts <= 5).sum() > 0:
    print(f'  5件以下クラス: {(counts <= 5).sum()} → pseudo-labeling で補完候補')

print(f'\n[Secondary labels]')
sec_ratio = has_sec / len(df)
print(f'  付与率: {sec_ratio:.1%}')
if sec_ratio > 0.1:
    print(f'  → secondary_label_weight=0.5 を使う価値あり (improved.yaml デフォルト)')
else:
    print(f'  → 付与率が低い。weight を下げるか無効化を検討')

print(f'\n[configs への反映推奨]')
print(f'  num_classes:  {n_cls}  (自動設定されるので明示不要)')
print(f'  use_focal_loss: true  (不均衡が {imb:.0f}x あるため)')
print(f'  secondary_label_weight: {0.5 if sec_ratio > 0.1 else 0.0}')